# Multi-provider calls, sampling controls, and temperature

**The messages format · temperature, top_p, max_tokens, stop · switching provider by editing .env**

Run top to bottom. Before starting: `git pull`, `uv sync` (no new packages today).

Two providers need a key: Groq (already in your `.env`) and Gemini (new today — see section 1). Ollama is optional; cells that need it skip cleanly if it is not running.


## 1. Setup — three clients, one function

A provider is a place; a model is a brain. `ask()` in `providers_utils.py` is the one function every provider boils down to: build a messages list, call `chat.completions.create`, read the text and the usage. Only the client and the model name change between providers.

**Getting a free Gemini key (2 minutes):** go to aistudio.google.com, sign in, click "Get API key", create one, and paste it into `.env` as `GEMINI_API_KEY`. No card needed.


In [1]:
import sys

sys.path[:0] = [".", "module01", "../module01"]
import os

from providers_utils import (
    ask,
    classify_ticket,
    gemini_client,
    groq_client,
    list_gemini_models,
    ollama_client,
)
from tickets_data import ALL_TICKETS, BORDERLINE_TICKETS, CLEAR_TICKETS  # noqa: F401

groq = groq_client()
GROQ_MODEL = os.getenv("MODEL")

try:
    gemini = gemini_client()
    # Gemini's Flash model name changes often - verify instead of trusting a name written down months ago.
    flash_models = list_gemini_models()
    GEMINI_MODEL = os.getenv("GEMINI_MODEL") or (
        flash_models[0] if flash_models else None
    )
    print("Gemini models available:", flash_models)
    print("using:", GEMINI_MODEL)
except Exception as e:  # noqa: BLE001
    gemini, GEMINI_MODEL = None, None
    print("Gemini not configured yet:", e)

try:
    ollama = ollama_client()
    OLLAMA_MODEL = "llama3.2:3b"
except Exception as e:  # noqa: BLE001
    ollama = None
    print("Ollama not available:", e)

Gemini models available: ['models/gemini-2.5-flash', 'models/gemini-2.5-flash-image', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-native-audio-latest', 'models/gemini-2.5-flash-native-audio-preview-09-2025', 'models/gemini-2.5-flash-native-audio-preview-12-2025', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-live-preview', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash', 'models/gemini-3.8-flash-lite-tts', 'models/gemini-3.8-flash-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-omni-1.1-flash', 'models/gemini-omni-flash-preview']
using: gemini-3.5-flash-lite


## 2. The messages format, properly

You have used this shape since `hello.py`: a list of role/content pairs. Two things worth seeing deliberately.

- The **system** message sets who the model is. Drop it and the model falls back to a generic default persona.
- **Order and role matter** — a system instruction buried as a second user message is weaker than a real system message.


In [2]:
system = "You are a terse assistant for a retail bank. Answer in one short sentence."
user = "What is an EMI?"

with_system = ask(groq, GROQ_MODEL, system, user)  # type: ignore
without_system = ask(groq, GROQ_MODEL, "", user)  # type: ignore

print("WITH system message:\n ", with_system["text"])
print("\nWITHOUT a system message:\n ", without_system["text"])

WITH system message:
  EMI is the fixed monthly payment you make to repay a loan.

WITHOUT a system message:
  **


_Question 1 — What changed between the two answers, and which field in `.env` would you edit to make this run against Gemini instead of Groq?_

**Ans:** The first answer was concise and on point. The second answer was broad, and has too much unnecessary info. I would comment out Groq variables.


## 3. Same prompt, three providers

Same system prompt, same question, three providers. Compare the text and the token usage. This is the entire point of today: the code around this cell does not change — only which client and model name go in.


In [3]:
prompt = "In one sentence, what is the difference between a workflow and an agent?"
results = {}
for label, client, model in [
    ("groq", groq, GROQ_MODEL),
    ("gemini", gemini, GEMINI_MODEL),
    ("ollama", ollama, "llama3.2:3b" if ollama else None),
]:
    # 1. Skip if client wasn't configured
    if client is None or model is None:
        print(f"--- {label}: skipped (not configured)\n")
        continue
    try:
        # 2. Universal call using the unified ask() function
        r = ask(client, model, "You are a concise AI engineering tutor.", prompt)
        results[label] = r

        # 3. Print output and token telemetry
        print(f"--- {label} ({model})")
        print(" ", r["text"].strip())
        print(
            f"   tokens: prompt={r['prompt_tokens']} completion={r['completion_tokens']} total={r['total_tokens']}\n"
        )
    except Exception as e:  # noqa: BLE001
        print(f"--- {label}: error ({type(e).__name__}: {str(e)[:100]})\n")

--- groq (gemini-3.5-flash-lite)
  A workflow follows a predefined, deterministic sequence of steps, whereas an agent dynamically decides its own actions and steps using an LLM based on the task and environment.
   tokens: prompt=25 completion=32 total=57

--- gemini (gemini-3.5-flash-lite)
  A **workflow** is a deterministic, hardcoded sequence of steps, whereas an **agent** is an autonomous system that uses an LLM to dynamically decide its own steps and tools to achieve a goal.
   tokens: prompt=25 completion=41 total=66

--- ollama (llama3.2:3b)
  A workflow is a series of tasks or processes that are executed in a specific order, whereas an agent is a software entity that autonomously interacts with the environment, making decisions and taking actions to achieve a goal.
   tokens: prompt=48 completion=43 total=91



_Question 2 — Which provider used the fewest total tokens for the same question? Does that necessarily mean it is the cheapest or the best choice — what else would you need to know (hint: Session 10)?_

**Ans:** Local Ollama used the fewest total tokens for the same question(groq = 175 vs gemini = 307 vs ollama(MY Laptop) = 91). For the given difficulty of question and the constraints, it is indeed the "cheapest and best choice", At first I thought I should give the award to groq because I used the API for free then I looked at the number of total tokens and realized that even if it's free it hits the limits quickly at this rate. If querys increase then Groq replaces Ollama.


## 4. Sampling: temperature, top_p, max_tokens, stop

- **temperature** (0 to roughly 2) controls how much randomness goes into picking the next token. 0 is as close to deterministic as a provider allows; higher values let the model pick less-likely tokens more often.
- **top_p** is an alternative way to control the same thing — restrict the choice to the smallest set of tokens whose probabilities add up to `top_p`. Most labs here use temperature alone; the two are rarely combined.
- **max_tokens** caps the length of the reply. Set it too low on a reasoning model and you can get an empty answer, as you saw in Session 10.
- **stop** ends generation the moment a given string appears — useful for forcing a short, structured answer.


In [4]:
short = ask(
    groq,
    GROQ_MODEL,  # type: ignore
    "You are a helpful assistant.",
    "Explain what a vector database is.",
    max_tokens=15,
)
enough = ask(
    groq,
    GROQ_MODEL,  # type: ignore
    "You are a helpful assistant.",
    "Explain what a vector database is.",
    max_tokens=150,
)
print("max_tokens=15:\n ", short["text"], "\n")
print("max_tokens=150:\n ", enough["text"][:300])

max_tokens=15:
   

max_tokens=150:
  ## What is a Vector Database?




In [5]:
r = ask(
    groq,
    GROQ_MODEL,  # type: ignore
    "You are a helpful assistant. Answer in a short paragraph.",
    "List three benefits of retrieval-augmented generation.",
    max_tokens=200,
    stop=["\n"],
)
print(r["text"])

Retrieval‑augmented generation (RAG) boosts language models by (1) **improving factual accuracy**—the model can pull in up‑to‑date, authoritative sources to ground its responses, reducing hallucinations; (2) **enhancing contextual relevance**—retrieved documents provide richer, domain‑specific context that guides generation toward more nuanced answers; and (3) **extending knowledge coverage**—the system can tap external corpora on demand, allowing it to answer


## 5. The temperature sweep — where it actually matters

Three clear tickets, three borderline ones. Run the same ticket through the classifier at several temperatures and watch what happens. A routing decision like this should usually be made at temperature 0 — you do not want "billing" vs "technical" to depend on a coin flip.


In [6]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]
RUNS_PER_TEMPERATURE = 3

if groq is None:
    print("skipped (no Groq client)")
else:
    for tid, text in BORDERLINE_TICKETS:
        print(f"\n=== {tid} ===\n{text}\n")
        for temp in TEMPERATURES:
            labels = []
            # 1. Run classifier at the given temperature
            for _ in range(RUNS_PER_TEMPERATURE):
                result = classify_ticket(groq, GROQ_MODEL, text, temperature=temp)  # type: ignore
                # 2. Extract label or mark invalid if it didn't match {"billing", "technical", "general"}
                labels.append(
                    result["label"]
                    if result["valid"]
                    else f"INVALID:{result['label']!r}"
                )
            # 3. Check stability: if all 3 runs produced the exact same label, len(set(labels)) == 1
            stable = len(set(labels)) == 1
            print(
                f"  temperature={temp:<4} {'stable' if stable else 'DRIFTED':8} {labels}"
            )


=== borderline_1 ===
I upgraded my plan but I'm still being charged the old price and the new features aren't showing up either.

  temperature=0.0  stable   ['billing', 'billing', 'billing']
  temperature=0.3  stable   ['billing', 'billing', 'billing']
  temperature=0.7  stable   ['billing', 'billing', 'billing']
  temperature=1.0  DRIFTED  ['billing', "INVALID:''", 'billing']

=== borderline_2 ===
Your app keeps logging me out and now I can't tell if I'm still being billed for the plan I signed up for.

  temperature=0.0  stable   ['billing', 'billing', 'billing']
  temperature=0.3  stable   ['billing', 'billing', 'billing']
  temperature=0.7  DRIFTED  ['billing', 'billing', "INVALID:''"]
  temperature=1.0  DRIFTED  ["INVALID:''", 'billing', 'billing']

=== borderline_3 ===
I'm not sure who to ask, but something seems wrong with my account since the last update.

  temperature=0.0  stable   ['technical', 'technical', 'technical']
  temperature=0.3  stable   ['technical', 'technical'

In [7]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]
RUNS_PER_TEMPERATURE = 3

if groq is None:
    print("skipped (no Groq client)")
else:
    for tid, text in CLEAR_TICKETS:
        print(f"\n=== {tid} ===\n{text}\n")
        for temp in TEMPERATURES:
            labels = []
            # 1. Run classifier at the given temperature
            for _ in range(RUNS_PER_TEMPERATURE):
                result = classify_ticket(groq, GROQ_MODEL, text, temperature=temp)  # type: ignore
                # 2. Extract label or mark invalid if it didn't match {"billing", "technical", "general"}
                labels.append(
                    result["label"]
                    if result["valid"]
                    else f"INVALID:{result['label']!r}"
                )
            # 3. Check stability: if all 3 runs produced the exact same label, len(set(labels)) == 1
            stable = len(set(labels)) == 1
            print(
                f"  temperature={temp:<4} {'stable' if stable else 'DRIFTED':8} {labels}"
            )


=== billing_clear ===
I was charged twice for my subscription this month. Please refund the duplicate charge.

  temperature=0.0  stable   ['billing', 'billing', 'billing']
  temperature=0.3  stable   ['billing', 'billing', 'billing']
  temperature=0.7  stable   ['billing', 'billing', 'billing']
  temperature=1.0  stable   ['billing', 'billing', 'billing']

=== technical_clear ===
The app crashes every time I try to upload a photo larger than 5MB.

  temperature=0.0  stable   ['technical', 'technical', 'technical']
  temperature=0.3  stable   ['technical', 'technical', 'technical']
  temperature=0.7  stable   ['technical', 'technical', 'technical']
  temperature=1.0  stable   ['technical', 'technical', 'technical']

=== general_clear ===
What are your customer support hours on weekends?

  temperature=0.0  stable   ['general', 'general', 'general']
  temperature=0.3  stable   ['general', 'general', 'general']
  temperature=0.7  stable   ['general', 'general', 'general']
  temperature=

_Question 3 — At which temperature, if any, did the label start to change between runs on the same ticket? What would you set temperature to for a routing decision in production, and why?_

**Ans:** The label started to change mostly from `0.3`, volitality started when lower probability logits started to gain percentages. For a production router use, I would set the temperature at `0.0`, because for a model like `gpt-oss-20b`, it would become too easily confused, just like we've seen in the above example. Having `0` temperature makes sure that the model doesn't hesitate on small fry and choses the logit with the highest percentage of probability. 

_Question 4 — Run the same sweep on one of the CLEAR_TICKETS. Does it drift as much as the borderline ones? What does that tell you about when temperature actually matters?_

**Ans:** There's not drifting at all when sweeping the `CLEAR_TICKETS`, because... they are 'CLEAR' tickets. Regarding temperature, it doesn't need to be higher than `0.0` for tasks like production routing, where raw categorization and classification is needed. Temperature only matters when there is ambiguity.

## 6. Switching provider - three lines, no code change

This is the payoff of provider abstraction. We will redirect the primary client (`BASE_URL`, `API_KEY`, `MODEL`) to talk directly to Gemini or Ollama without changing a single line of Python code.

---

### Step-by-Step Instructions:

#### 1. Edit your `.env` file:
Comment out Groq and point the primary variables to Gemini (or local Ollama). Make sure `API_KEY` contains your actual key so the client doesn't complain:

```env
# Point the generic provider at Gemini:
BASE_URL=https://generativelanguage.googleapis.com/v1beta/openai/
API_KEY=your_actual_gemini_key_here
MODEL=gemini-2.5-flash-lite

# Keep these in place so gemini_client() stays valid:
GEMINI_API_KEY=your_actual_gemini_key_here
GEMINI_MODEL=gemini-2.5-flash-lite
```

#### 2. Execution Order (Avoid the 15 RPM Rate Limit!):

IMPORTANT: Do NOT re-run Section 5's temperature sweep! That cell fires 36 rapid API calls, which immediately blows past Google AI Studio's 15 RPM (Requests Per Minute) free-tier limit and triggers a 429 error.

##### Follow this exact order:

- Restart the Kernel: In the top menu, click Kernel > Restart Kernel.
- Run Section 1: Executes the setup handshake and verifies the new environment.
- Run this cell (Section 6): Confirms that BASE_URL now: and MODEL now: reflect your new provider.
- Run Section 3: Sends the single question to the new provider and compares token usage and output.

#### 3. Cleanup:

When done testing, restore your .env back to Groq as the default provider before running the final tests in Section 7.

In [2]:
# After editing .env, restart the kernel (Kernel > Restart), then re-run section 1's setup cell
# and this cell to see the SAME code now talking to a different provider.
import importlib

import providers_utils

importlib.reload(providers_utils)
from providers_utils import ask

print("BASE_URL now:", os.getenv("BASE_URL"))
print("MODEL now:", os.getenv("MODEL"))

BASE_URL now: https://generativelanguage.googleapis.com/v1beta/openai/
MODEL now: gemini-3.5-flash-lite


## 7. Reflection

Answer questions 1–4 above, one or two lines each.

_**Question 1** — What changed between the two answers, and which field in `.env` would you edit to make this run against Gemini instead of Groq?_

**Ans:** The first answer was concise and on point. The second answer was broad, and has too much unnecessary info. I would comment out Groq variables.

_**Question 2** — Which provider used the fewest total tokens for the same question? Does that necessarily mean it is the cheapest or the best choice — what else would you need to know (hint: Session 10)?_

**Ans:** Local Ollama used the fewest total tokens for the same question(groq = 175 vs gemini = 307 vs ollama(MY Laptop) = 91). For the given difficulty of question and the constraints, it is indeed the "cheapest and best choice", At first I thought I should give the award to groq because I used the API for free then I looked at the number of total tokens and realized that even if it's free it hits the limits quickly at this rate. If querys increase then Groq replaces Ollama.

_**Question 3** — At which temperature, if any, did the label start to change between runs on the same ticket? What would you set temperature to for a routing decision in production, and why?_

**Ans:** The label started to change mostly from `0.3`, volitality started when lower probability logits started to gain percentages. For a production router use, I would set the temperature at `0.0`, because for a model like `gpt-oss-20b`, it would become too easily confused, just like we've seen in the above example. Having `0` temperature makes sure that the model doesn't hesitate on small fry and choses the logit with the highest percentage of probability. 

_**Question 4** — Run the same sweep on one of the CLEAR_TICKETS. Does it drift as much as the borderline ones? What does that tell you about when temperature actually matters?_

**Ans:** There's not drifting at all when sweeping the `CLEAR_TICKETS`, because... they are 'CLEAR' tickets. Regarding temperature, it doesn't need to be higher than `0.0` for tasks like production routing, where raw categorization and classification is needed. Temperature only matters when there is ambiguity.

Then run the checkpoint from the terminal: `uv run pytest tests/test_s11.py`
